In [1]:
import kagglehub

path = kagglehub.dataset_download("palashfendarkar/wa-fnusec-telcocustomerchurn")

print("Path to dataset files:", path)

100%|████████████████████████████████████████████████████████████████████████████████| 172k/172k [00:00<00:00, 247kB/s]

Extracting files...
Path to dataset files: C:\Users\User\.cache\kagglehub\datasets\palashfendarkar\wa-fnusec-telcocustomerchurn\versions\1


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

filepath = r"C:\Users\User\.cache\kagglehub\datasets\palashfendarkar\wa-fnusec-telcocustomerchurn\versions\1\WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(filepath)

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df.drop('customerID', axis=1, inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [col for col in X.columns if col not in numeric_features]

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', RandomForestClassifier(random_state=42))])

param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [5, 10, None]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")

y_pred = grid_search.best_estimator_.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred))

joblib.dump(grid_search.best_estimator_, 'telco_churn_pipeline.pkl')

Best Parameters: {'classifier__max_depth': 10, 'classifier__n_estimators': 200}
Accuracy: 0.805
              precision    recall  f1-score   support

           0       0.84      0.91      0.87      1036
           1       0.67      0.52      0.58       373

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.73      1409
weighted avg       0.79      0.80      0.80      1409



['telco_churn_pipeline.pkl']